# ViT-B/16 Training Notebook - Binary Chest X-Ray Classification

This notebook mirrors EfficientNet reference notebook, but uses shared ViT training pipeline in `pytorch_engine.chest_xray_vit_training`.

Use it when you want:
- clean ViT transfer-learning reference for medical images
- readable explanation of why ViT preprocessing differs from EfficientNet
- two-phase training without notebook-local training-loop clutter
- checkpointed, reproducible notebook workflow
        


## Why This Pipeline Looks Like This

ViT-B/16 needs slightly different handling than EfficientNet:
- keep torchvision 224px pretrained setup so positional embeddings stay aligned
- use chest-X-ray-safe resize/crop policy, not natural-image random flips
- warm up fresh head before selectively unfreezing final encoder blocks
- use AdamW with lower backbone LR and no-decay groups for norms, biases, tokens, and positional embeddings
- apply MixUp only during fine-tuning, where extra regularization helps most
- use validation AUROC for early stopping and best-checkpoint selection
        


## 1. Imports And Project Bootstrap


In [ ]:
import logging
import os
import sys
from pathlib import Path

candidate_roots: list[Path] = []
engine_root_override = os.getenv("PYTORCH_ENGINE_ROOT")
if engine_root_override:
    candidate_roots.append(Path(engine_root_override).expanduser().resolve())

cwd = Path.cwd().resolve()
candidate_roots.extend([cwd, *cwd.parents])

ENGINE_ROOT: Path | None = None
for candidate in candidate_roots:
    pyproject_path = candidate / "pyproject.toml"
    if not pyproject_path.is_file():
        continue
    try:
        if 'name = "pytorch-engine"' in pyproject_path.read_text(encoding="utf-8"):
            ENGINE_ROOT = candidate
            break
    except OSError:
        continue

if ENGINE_ROOT is None:
    raise FileNotFoundError(
        "Could not find pytorch-engine root. Set PYTORCH_ENGINE_ROOT or start notebook inside app directory."
    )

SRC_ROOT = ENGINE_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
        


In [ ]:
import pandas as pd
import torch
from dotenv import load_dotenv
from pytorch_engine import (
    ChestXrayVitTrainingConfig,
    build_chest_xray_vit_imagefolder_loaders,
    create_milestone_checkpoint_callback,
    download_and_prepare_kaggle_chest_xray_pneumonia_dataset,
    find_cross_split_duplicate_files,
    get_current_device,
    run_chest_xray_vit_training,
    summarize_imagefolder_splits,
)
from pytorch_engine.models import create_vit_model
from pytorch_engine.utils import configure_torch_runtime
from pytorch_engine.visualization import plot_confusion_matrix, plot_loss_curves
from torchinfo import summary

_ = load_dotenv(ENGINE_ROOT / ".env")
        


## 2. Runtime And Training Configuration

Most users only need to change `DATASET_ROOT`, `OUTPUT_DIR`, or a few schedule hyperparameters.

Design choices:
- keep `image_size=(224, 224)` because torchvision ViT-B/16 weights expect that patch geometry
- Windows notebooks default to `num_workers=0` for stability
- CUDA runs enable AMP, TF32, and optional `torch.compile`
- milestone checkpoint callback stays optional so local iteration stays fast
        


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True,
)

device = get_current_device()
configure_torch_runtime(device)
AMP_DTYPE = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

if device.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capability = torch.cuda.get_device_capability(0)
    free_memory_bytes, total_memory_bytes = torch.cuda.mem_get_info()
    use_tf32 = bool(torch.backends.cuda.matmul.allow_tf32)
else:
    gpu_name = "cpu"
    gpu_capability = None
    free_memory_bytes = None
    total_memory_bytes = None
    use_tf32 = False

DATASET_ROOT = Path(
    os.getenv(
        "CHEST_XRAY_DATASET_ROOT",
        str(ENGINE_ROOT / "data" / "chest-xray-pneumonia-balanced-dataset"),
    )
)
DATASET_ROOT = download_and_prepare_kaggle_chest_xray_pneumonia_dataset(destination=DATASET_ROOT)
OUTPUT_DIR = Path(
    os.getenv(
        "CHEST_XRAY_OUTPUT_DIR",
        str(ENGINE_ROOT / "src" / "pytorch-saved-models" / "vitb16_chest_xray"),
    )
)

NUM_WORKERS = 0 if os.name == "nt" else None
BATCH_SIZE = int(os.getenv("CHEST_XRAY_BATCH_SIZE", "16" if device.type == "cuda" else "8"))

config = ChestXrayVitTrainingConfig(
    dataset_root=DATASET_ROOT,
    output_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_amp=device.type == "cuda",
    amp_dtype=AMP_DTYPE,
    use_channels_last=False,
    compile_model=device.type == "cuda",
    compile_mode="max-autotune",
)

runtime_summary = pd.DataFrame(
    [
        {
            "setting": "dataset_root",
            "value": str(config.dataset_root),
            "notes": "Downloaded on demand if missing; existing prepared dataset is reused.",
        },
        {
            "setting": "output_dir",
            "value": str(config.output_dir),
            "notes": "Best and last checkpoints are written here.",
        },
        {
            "setting": "device",
            "value": str(device),
            "notes": "CUDA enables AMP, TF32, and optional torch.compile.",
        },
        {
            "setting": "batch_size",
            "value": config.batch_size,
            "notes": "16 is conservative default for ViT-B/16 on T4-class GPUs.",
        },
        {
            "setting": "phase_schedule",
            "value": f"warmup={config.head_warmup_epochs}, fine_tune={config.fine_tune_epochs}",
            "notes": "Head warmup stabilizes training before selective encoder unfreezing.",
        },
        {
            "setting": "mixup",
            "value": f"alpha={config.mixup_alpha}, p={config.mixup_probability}",
            "notes": "MixUp only runs during fine-tuning for extra small-data regularization.",
        },
        {
            "setting": "selection_metric",
            "value": config.selection_metric,
            "notes": "Validation AUROC drives early stopping and best-checkpoint restore.",
        },
    ]
)
display(runtime_summary)

if device.type == "cuda":
    print(f"GPU: {gpu_name}")
    print(f"Capability: {gpu_capability}")
    print(f"GPU memory: {free_memory_bytes / 1e9:.2f} GB free / {total_memory_bytes / 1e9:.2f} GB total")
    print(f"AMP dtype: {AMP_DTYPE}")
    print(f"TF32 enabled: {use_tf32}")
else:
    print("CUDA not available. Notebook will run on CPU, but full ViT fine-tuning is tuned for GPU.")
        


## 2a. Optional Milestone Checkpoints For Long Runs

Local notebook work usually only needs final `best.pth` and `last.pth` checkpoints.

If you move to a long remote run, uncomment next cell to save milestone checkpoints and optionally upload them to Hugging Face.
        


In [ ]:
# HF_USERNAME = os.getenv("HF_USERNAME")
# HF_TOKEN = os.getenv("HF_TOKEN")
# HF_REPO_SUBDIR = "vitb16_chest_xray"
# HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_SUBDIR}" if HF_USERNAME else None
# checkpoint_callback = create_milestone_checkpoint_callback(
#     model_name_prefix="vitb16_chest_xray",
#     total_epochs=config.fine_tune_epochs,
#     every_n_epochs=5,
#     target_dir=str(config.output_dir),
#     hf_repo_id=HF_REPO_ID,
#     hf_token=HF_TOKEN,
#     hf_repo_subdir=HF_REPO_SUBDIR,
#     upload_best_checkpoint=False,
# )
# config.fine_tune_epoch_end_callback = checkpoint_callback
        


## 3. Dataset And Preprocessing Sanity Check

ViT is sensitive to preprocessing drift. This section confirms split structure, checks for exact duplicate files across splits, and previews loader policy before training starts.
        


In [ ]:
expected_dirs = [
    DATASET_ROOT / split_name / class_name
    for split_name in ("train", "val", "test")
    for class_name in ("NORMAL", "PNEUMONIA")
]
missing_dirs = [path for path in expected_dirs if not path.is_dir()]
if missing_dirs:
    raise FileNotFoundError(
        "Dataset structure is incomplete. Missing directories:\n" + "\n".join(str(path) for path in missing_dirs)
    )

split_summary = pd.DataFrame(summarize_imagefolder_splits(DATASET_ROOT)).set_index("split")
split_summary["pneumonia_rate"] = split_summary["PNEUMONIA"] / split_summary["total"]
display(split_summary.round(4))

cross_split_duplicates = find_cross_split_duplicate_files(DATASET_ROOT)
duplicate_rows = [
    {
        "sha256": duplicate_group["sha256"][:12],
        "split": file_record["split"],
        "path": file_record["path"],
    }
    for duplicate_group in cross_split_duplicates
    for file_record in duplicate_group["files"]
]
if duplicate_rows:
    print("Warning: exact duplicate files found across dataset splits. Treat test metrics as optimistic.")
    display(pd.DataFrame(duplicate_rows))
else:
    print("No exact duplicate files detected across train/val/test splits.")

preview_loaders = build_chest_xray_vit_imagefolder_loaders(config)
loader_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "batches": len(preview_loaders.train_dataloader),
            "batch_size": preview_loaders.train_dataloader.batch_size,
            "drop_last": preview_loaders.train_dataloader.drop_last,
            "pin_memory": preview_loaders.train_dataloader.pin_memory,
        },
        {
            "split": "val",
            "batches": len(preview_loaders.val_dataloader),
            "batch_size": preview_loaders.val_dataloader.batch_size,
            "drop_last": preview_loaders.val_dataloader.drop_last,
            "pin_memory": preview_loaders.val_dataloader.pin_memory,
        },
        {
            "split": "test",
            "batches": len(preview_loaders.test_dataloader),
            "batch_size": preview_loaders.test_dataloader.batch_size,
            "drop_last": preview_loaders.test_dataloader.drop_last,
            "pin_memory": preview_loaders.test_dataloader.pin_memory,
        },
    ]
).set_index("split")
display(loader_summary)
print(f"Class order: {preview_loaders.class_names}")
print("\nTrain transform:\n", preview_loaders.train_dataloader.dataset.transform)
print("\nEval transform:\n", preview_loaders.val_dataloader.dataset.transform)
        


## 4. Optional Pre-Training Model Summary

This preview confirms head shape and trainable parameter count before training. Summary runs on CPU so it does not steal GPU memory from real training.
        


In [ ]:
model_preview = create_vit_model(
    num_classes=2,
    seed=config.seed,
    trainable_encoder_blocks=0,
).model

preview_trainable_params = sum(parameter.numel() for parameter in model_preview.parameters() if parameter.requires_grad)
preview_total_params = sum(parameter.numel() for parameter in model_preview.parameters())
print(f"Trainable params before fine-tuning: {preview_trainable_params:,} / {preview_total_params:,}")

summary(
    model_preview,
    input_size=(1, 3, *config.image_size),
    col_names=("input_size", "output_size", "num_params", "trainable"),
    row_settings=("var_names",),
    device="cpu",
)

del model_preview
        


## 5. Train ViT-B/16

Shared pipeline will:
- build ViT-aligned train/val/test loaders
- warm up classifier head
- unfreeze final encoder blocks plus final normalization, class token, and positional embedding
- fine-tune with AdamW, MixUp, LR warmup, cosine decay, and AUROC-based early stopping
- keep stable local `best.pth` and `last.pth` checkpoints
        


In [ ]:
run = run_chest_xray_vit_training(config)


## 6. Review Curves, Metrics, And Checkpoints

Mixed-phase history makes it easier to see whether head warmup stabilized optimization before selective backbone fine-tuning started.
        


In [ ]:
merged_results = {
    "train_loss": [],
    "test_loss": [],
    "train_acc": [],
    "test_acc": [],
}
for phase_results in (run.warmup_results, run.fine_tune_results):
    if phase_results is None:
        continue
    for metric_name in merged_results:
        merged_results[metric_name].extend(phase_results[metric_name])

optimizer_group_summary = pd.DataFrame(
    [
        {
            "group": summary_row.group_name,
            "lr": summary_row.learning_rate,
            "weight_decay": summary_row.weight_decay,
            "parameters": summary_row.parameter_count,
        }
        for summary_row in run.optimizer_group_summaries
    ]
).set_index("group") if run.optimizer_group_summaries else pd.DataFrame()
if not optimizer_group_summary.empty:
    display(optimizer_group_summary)

phase_summary = pd.DataFrame(
    [
        {
            "phase": "head_warmup",
            "epochs": len(run.warmup_results["train_loss"]) if run.warmup_results is not None else 0,
            "val_accuracy": None if run.warmup_val_metrics is None else run.warmup_val_metrics["accuracy"],
            "val_macro_f1": None if run.warmup_val_metrics is None else run.warmup_val_metrics["macro_f1"],
            "val_auroc": None if run.warmup_val_metrics is None else run.warmup_val_metrics["auroc"],
        },
        {
            "phase": "selected_final_model",
            "epochs": len(merged_results["train_loss"]),
            "val_accuracy": run.val_metrics["accuracy"],
            "val_macro_f1": run.val_metrics["macro_f1"],
            "val_auroc": run.val_metrics["auroc"],
        },
    ]
).set_index("phase")
display(phase_summary.round(4))

split_metrics = pd.DataFrame(
    [
        {
            "split": "validation",
            "accuracy": run.val_metrics["accuracy"],
            "balanced_accuracy": run.val_metrics["balanced_accuracy"],
            "macro_f1": run.val_metrics["macro_f1"],
            "macro_precision": run.val_metrics["macro_precision"],
            "macro_recall": run.val_metrics["macro_recall"],
            "weighted_f1": run.val_metrics["weighted_f1"],
            "auroc": run.val_metrics["auroc"],
            "average_precision": run.val_metrics["average_precision"],
        },
        {
            "split": "test",
            "accuracy": run.test_metrics["accuracy"],
            "balanced_accuracy": run.test_metrics["balanced_accuracy"],
            "macro_f1": run.test_metrics["macro_f1"],
            "macro_precision": run.test_metrics["macro_precision"],
            "macro_recall": run.test_metrics["macro_recall"],
            "weighted_f1": run.test_metrics["weighted_f1"],
            "auroc": run.test_metrics["auroc"],
            "average_precision": run.test_metrics["average_precision"],
        },
    ]
).set_index("split")
display(split_metrics.round(4))

per_class_metrics = pd.DataFrame(
    {
        "precision": run.test_metrics["per_class_precision"],
        "recall": run.test_metrics["per_class_recall"],
        "f1": run.test_metrics["per_class_f1"],
        "support": run.test_metrics["per_class_support"],
    }
).loc[run.class_names]
display(per_class_metrics.round(4))

print(f"Selected phase: {run.selected_phase}")
print(f"Unfrozen encoder blocks: {run.unfrozen_encoder_blocks}")
print(f"Best checkpoint: {run.best_checkpoint_path}")
print(f"Last checkpoint: {run.last_checkpoint_path}")

if merged_results["train_loss"]:
    curves_figure = plot_loss_curves(merged_results)
    curves_figure.show()

confusion_figure = plot_confusion_matrix(
    confusion_matrix_values=run.test_metrics["normalized_confusion_matrix"],
    class_names=run.class_names,
    normalize=True,
)
confusion_figure.show()
        


## 7. Final Model Summary

Inspect final restored model after training. This confirms head shape after best-checkpoint restore.
        


In [ ]:
summary(
    run.model,
    input_size=(1, 3, *config.image_size),
    col_names=("input_size", "output_size", "num_params", "trainable"),
    row_settings=("var_names",),
    device=str(device),
)
        
